<a href="https://colab.research.google.com/github/Alberto-UTNAY/Modulo_5_Encodering_Segment/blob/main/M5_2_Autoencoder_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torchinfo

In [ ]:
import torch
import torch.nn as nn
from torchinfo import summary
torch.manual_seed(0)

In [ ]:
def compression_factor(depth, in_channels, latent_channels, spatial_dims):
    # each axis shrinks by axis_reduction (e.g. depth=2 -> axis_reduction=4)
    axis_reduction = 2 ** depth
    # spatial_dims axes shrink at the same time, so the element count compounds: axis_reduction**spatial_dims
    spatial_factor = axis_reduction ** spatial_dims
    channel_factor = in_channels / latent_channels
    return axis_reduction, spatial_factor, channel_factor, spatial_factor * channel_factor


def print_compression(name, depth, in_channels, latent_channels, spatial_dims):
    """Print the spatial vs channel breakdown; torchinfo.summary already covers shapes and params."""
    axis, spatial, channel, total = compression_factor(depth, in_channels, latent_channels, spatial_dims)
    print(f"{name} compression factor (depth={depth}): axis /{axis:.0f} -> spatial x{spatial:.2f} (elements) * channels x{channel:.2f} = x{total:.2f}")


def show_architecture(ae, x):
    """Print the full layer-by-layer architecture, input/output shapes and parameter counts."""
    _ = summary(ae, input_data=x, col_names=("input_size", "output_size", "num_params"), verbose=1, depth=3)

In [ ]:
class DownBlock1D(nn.Module):
    """MLP encoder step: merge pairs of samples, halving the length."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(2 * in_channels, out_channels),
            nn.ReLU(inplace=True),
            nn.LayerNorm(out_channels),
        )

    def forward(self, x):
        # x: [batch, channels, length]
        batch, channels, length = x.shape
        if length % 2 != 0:
            raise ValueError("DownBlock1D expects an even length.")
        x = x.transpose(1, 2)                    # [batch, length, channels]
        x = x.reshape(batch, length // 2, 2 * channels)
        x = self.block(x)                        # [batch, length // 2, out_channels]
        return x.transpose(1, 2)                 # [batch, out_channels, length // 2]


class UpBlock1D(nn.Module):
    """MLP decoder step: split each sample into two, doubling the length."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.out_channels = out_channels
        self.proj = nn.Linear(in_channels, 2 * out_channels)
        self.act = nn.ReLU(inplace=True)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        # x: [batch, channels, length]
        batch, _, length = x.shape
        x = x.transpose(1, 2)                    # [batch, length, channels]
        x = self.act(self.proj(x))               # [batch, length, 2 * out_channels]
        x = x.reshape(batch, length * 2, self.out_channels)
        x = self.norm(x)
        return x.transpose(1, 2)                 # [batch, out_channels, length * 2]


In [ ]:
class Autoencoder1D(nn.Module):
    """Stacks `depth` DownBlock/UpBlock pairs."""
    def __init__(self, in_channels, latent_channels, depth=1):
        super().__init__()
        self.depth = depth
        channels = [in_channels] + [latent_channels] * depth
        self.encoder = nn.ModuleList(
            [DownBlock1D(channels[i], channels[i + 1]) for i in range(depth)]
        )
        self.decoder = nn.ModuleList(
            [UpBlock1D(channels[i + 1], channels[i]) for i in reversed(range(depth))]
        )

    def encode(self, x):
        z = x
        for down in self.encoder:
            z = down(z)
        return z

    def decode(self, z):
        y = z
        for up in self.decoder:
            y = up(y)
        return y

    def forward(self, x):
        return self.decode(self.encode(x))


x1 = torch.randn(4, 1, 1024)

ae1 = Autoencoder1D(in_channels=1, latent_channels=32, depth=10)
show_architecture(ae1, x1)
print_compression("1D", depth=10, in_channels=2, latent_channels=2, spatial_dims=1)

#ae1_deep = Autoencoder1D(in_channels=3, latent_channels=2, depth=2)
#show_architecture(ae1, x1)
#print_compression("1D (depth=2)", depth=2, in_channels=3, latent_channels=2, spatial_dims=1)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
Autoencoder1D                            [4, 1, 1024]              [4, 1, 1024]              --
├─ModuleList: 1-1                        --                        --                        --
│    └─DownBlock1D: 2-1                  [4, 1, 1024]              [4, 32, 512]              --
│    │    └─Sequential: 3-1              [4, 512, 2]               [4, 512, 32]              160
│    └─DownBlock1D: 2-2                  [4, 32, 512]              [4, 32, 256]              --
│    │    └─Sequential: 3-2              [4, 256, 64]              [4, 256, 32]              2,144
│    └─DownBlock1D: 2-3                  [4, 32, 256]              [4, 32, 128]              --
│    │    └─Sequential: 3-3              [4, 128, 64]              [4, 128, 32]              2,144
│    └─DownBlock1D: 2-4                  [4, 32, 128]              [4, 32, 64]               --
│    │    └─Sequential: 3-4 

# **Autoencoder 2D**

In [ ]:
class ConvBlock2D(nn.Module):
    """Conv -> Activation -> Norm, spatial size unchanged (stride=1)."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        return self.block(x)


class DownBlock2D(nn.Module):
    """Encoder step: ConvBlock followed by pooling, which halves height and width."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = ConvBlock2D(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        return self.pool(self.conv(x))


class UpBlock2D(nn.Module):
    """Decoder step: upsampling (linear/bilinear or ConvTranspose) followed by ConvBlock."""
    def __init__(self, in_channels, out_channels, mode="convtranspose"):
        super().__init__()
        if mode == "convtranspose":
            self.upsample = nn.ConvTranspose2d(in_channels, in_channels, kernel_size=2, stride=2)
        elif mode == "linear":
            self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        else:
            raise ValueError(f"Unknown upsampling mode: {mode}")
        self.conv = ConvBlock2D(in_channels, out_channels)

    def forward(self, x):
        return self.conv(self.upsample(x))

In [ ]:
class Autoencoder2D(nn.Module):
    """Stacks `depth` DownBlock/UpBlock pairs."""
    def __init__(self, in_channels, latent_channels, depth=1, up_mode="convtranspose"):
        super().__init__()
        # TODO: Implement the encoder and decoder based on depth attribute use nn.ModuleList
        channels = [in_channels] + [latent_channels] * depth
        self.encoder = nn.ModuleList(
            [DownBlock2D(channels[i], channels[i + 1]) for i in range(depth)]
        )
        self.decoder = nn.ModuleList(
            [UpBlock2D(channels[i + 1], channels[i]) for i in reversed(range(depth))]
        )

    def encode(self, x):
        z = x
        for down in self.encoder:
            z = down(z)
        return z

    def decode(self, z):
        y = z
        for up in self.decoder:
            y = up(y)
        return y

    def forward(self, x):
        return self.decode(self.encode(x))

x2 = torch.randn(2, 3, 256, 256)

ae2 = Autoencoder2D(in_channels=3, latent_channels=4, depth=6)
show_architecture(ae2, x2)
print_compression("2D", depth=3, in_channels=3, latent_channels=4, spatial_dims=2)

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
Autoencoder2D                                 [2, 3, 256, 256]          [2, 3, 256, 256]          --
├─ModuleList: 1-1                             --                        --                        --
│    └─DownBlock2D: 2-1                       [2, 3, 256, 256]          [2, 4, 128, 128]          --
│    │    └─ConvBlock2D: 3-1                  [2, 3, 256, 256]          [2, 4, 256, 256]          120
│    │    └─MaxPool2d: 3-2                    [2, 4, 256, 256]          [2, 4, 128, 128]          --
│    └─DownBlock2D: 2-2                       [2, 4, 128, 128]          [2, 4, 64, 64]            --
│    │    └─ConvBlock2D: 3-3                  [2, 4, 128, 128]          [2, 4, 128, 128]          156
│    │    └─MaxPool2d: 3-4                    [2, 4, 128, 128]          [2, 4, 64, 64]            --
│    └─DownBlock2D: 2-3                       [2, 4, 64, 64]            [2, 4, 32, 3

# **Autoencoder 3D**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class DownBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, pool_size=2):  # ← Cambiar a 2 por defecto
        super().__init__()
        self.conv = ConvBlock3D(in_channels, out_channels)
        self.pool = nn.MaxPool3d(kernel_size=pool_size, stride=pool_size)  # ← Más manejable

    def forward(self, x):
        x = self.conv(x)
        x = self.pool(x)
        return x

class UpBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, mode="convtranspose", up_size=2):  # ← Cambiar a 2
        super().__init__()
        if mode == "convtranspose":
            self.up = nn.ConvTranspose3d(in_channels, out_channels, kernel_size=up_size, stride=up_size)
        else:
            self.up = nn.Upsample(scale_factor=up_size, mode='trilinear', align_corners=False)

        self.conv = ConvBlock3D(out_channels, out_channels)

    def forward(self, x):
        x = self.up(x)
        x = self.conv(x)
        return x

class Autoencoder3D(nn.Module):
    def __init__(self, in_channels, latent_channels, depth=1, up_mode="convtranspose"):
        super().__init__()

        # Encoder
        self.encoder = nn.ModuleList()
        channels = in_channels
        for _ in range(depth):
            self.encoder.append(DownBlock3D(channels, channels * 2, pool_size=2))
            channels *= 2

        # Bottleneck
        self.bottleneck = nn.Conv3d(channels, latent_channels, kernel_size=1)

        # Decoder
        self.decoder = nn.ModuleList()
        for i in range(depth):
            in_ch = latent_channels if i == 0 else channels
            out_ch = in_channels if i == depth - 1 else in_channels * (2 ** (depth - i - 1))
            self.decoder.append(UpBlock3D(in_ch, out_ch, mode=up_mode, up_size=2))
            channels = out_ch

    def forward(self, x):
        skips = []
        for down in self.encoder:
            x = down(x)
            skips.append(x)

        x = self.bottleneck(x)

        for up, skip in zip(self.decoder, skips[::-1]):
            x = up(x)
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[-3:], mode='trilinear', align_corners=False)
            x = x + skip

        return x

# Test con torchinfo
from torchinfo import summary

x3 = torch.randn(2, 1, 28, 28, 28)
ae3 = Autoencoder3D(in_channels=1, latent_channels=4, depth=1)

# Probar forward manual primero
#print("Probando forward manual...")
#try:
#    output = ae3(x3)
#    print(f"✓ Forward exitoso! Output shape: {output.shape}")
#except Exception as e:
#    print(f"✗ Error en forward: {e}")

# Luego probar torchinfo
#print("\nProbando torchinfo...")
#try:
#    summary(ae3, input_size=(2, 1, 28, 28, 28))
#    print("✓ torchinfo exitoso!")
#except Exception as e:
#    print(f"✗ Error en torchinfo: {e}")

In [ ]:
class Autoencoder3D(nn.Module):
    """Stacks `depth` DownBlock/UpBlock pairs."""
    def __init__(self, in_channels, latent_channels, depth=1, up_mode="convtranspose"):
        super().__init__()
        # TODO: Implement the encoder and decoder based on depth attribute use nn.ModuleList
        channels = [in_channels] + [latent_channels] * depth
        self.encoder = nn.ModuleList(
            [DownBlock3D(channels[i], channels[i + 1]) for i in range(depth)]
        )
        self.decoder = nn.ModuleList(
            [UpBlock3D(channels[i + 1], channels[i]) for i in reversed(range(depth))]
        )

    def encode(self, x):
        z = x
        for down in self.encoder:
            z = down(z)
        return z

    def decode(self, z):
        y = z
        for up in self.decoder:
            y = up(y)
        return y

    def forward(self, x):
        return self.decode(self.encode(x))

x3 = torch.randn(2, 1, 32, 32, 32)

ae3 = Autoencoder3D(in_channels=1, latent_channels=8, depth=3)
show_architecture(ae3, x3)
print_compression("3D", depth=3, in_channels=1, latent_channels=32, spatial_dims=2)